In [1]:
import warnings
import lightning as L
import yaml

from src.segmentators import Segmentator
from src.models import get_model
from src.datamodules import HFDataModule
from src.config import TrainConfig

warnings.filterwarnings(
    "ignore",
    message="You called `self.log",
)

L.seed_everything(42)


def validate_checkpoint(checkpoint_to_validate: str, config_path: str):
    with open(config_path, "r") as f:
        config = yaml.safe_load(f)

    cfg = TrainConfig(**config)
    model = get_model(cfg.model, **cfg.model_kwargs.model_dump())

    train_path = f"{cfg.dataset_name}_HF/{cfg.dataset_name}_train_patches-{cfg.patch_size}x{cfg.patch_size}/*"
    val_path = f"{cfg.dataset_name}_HF/{cfg.dataset_name}_validation/*"

    segmentation_model = Segmentator.load_from_checkpoint(
        checkpoint_to_validate,
        model=model,
        n_classes=cfg.n_classes,
        criterion=cfg.get_loss(),
        batch_size=cfg.batch_size,
        patch_size=cfg.patch_size,
        overlap=cfg.overlap,
        lr=cfg.lr,
        weight_decay=cfg.weight_decay,
        weights_only=False,
    )

    dm = HFDataModule(
        train_path=train_path,
        val_path=val_path,
        batch_size=cfg.batch_size,
    )

    trainer = L.Trainer(
        enable_model_summary=False,
        max_epochs=cfg.max_epochs,
        accelerator="gpu",
        devices=1,
        enable_progress_bar=True,
        accumulate_grad_batches=cfg.grad_accumulation_batches,
        log_every_n_steps=1,
        precision=cfg.precision,
        logger=False,
    )

    output = trainer.validate(
        model=segmentation_model,
        datamodule=dm,
    )
    return output


Seed set to 42


In [2]:
CHECKPOINT_TO_VALIDATE = "l_checkpoints/Vaihingen/segnet/v1/epoch=00-step=9-val_iou=0.088-val_f1=0.149-val_loss=0.688.ckpt"
CONFIG_PATH = "config_files/segnet_vaihingen_v1.yaml"

output = validate_checkpoint(CHECKPOINT_TO_VALIDATE, CONFIG_PATH)

Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/home/datacuber/Documents/semantic_segmentation/.venv/lib/python3.13/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃      Validate metric      ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│       val_accuracy        │    0.21202313899993896    │
│          val_f1           │    0.1491287797689438     │
│          val_iou          │    0.08809343725442886    │
│         val_loss          │     0.688334047794342     │
│       val_precision       │    0.21839427947998047    │
│        val_recall         │    0.21202313899993896    │
└───────────────────────────┴───────────────────────────┘